In [ ]:
!pip install -q pyspark pyarrow pandas

In [ ]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    lit,
    count,
    when
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
spark = (
    SparkSession.builder
    .appName("TechChallenge_Silver")
    .getOrCreate()
)

In [ ]:
BASE_PATH = "/content/drive/MyDrive/TechChallenge_Fase2"

DATA_PATH = f"{BASE_PATH}/data"

BRONZE_PATH = f"{DATA_PATH}/bronze"

SILVER_PATH = f"{DATA_PATH}/silver_bigquery"

In [ ]:
TABELAS_BRONZE = [
    "municipio",
    "uf",
    "meta_municipio",
    "meta_uf",
    "meta_brasil",
    "alunos_agregados"
]

bronze = {
    nome: spark.read.parquet(f"{BRONZE_PATH}/{nome}")
    for nome in TABELAS_BRONZE
}

for nome, df in bronze.items():
    print(
        f"{nome}: {df.count():,} registros "
        f"e {len(df.columns)} colunas"
    )

municipio: 23,995 registros e 19 colunas
uf: 145 registros e 19 colunas
meta_municipio: 10,704 registros e 17 colunas
meta_uf: 81 registros e 16 colunas
meta_brasil: 3 registros e 15 colunas
alunos_agregados: 79,273 registros e 17 colunas


In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col,
    isnan,
    lower,
    trim,
    when
)


class SilverProcessor:


    @staticmethod
    def remover_duplicados(df: DataFrame) -> DataFrame:
        return df.dropDuplicates()

    @staticmethod
    def normalizar_textos(
        df: DataFrame,
        colunas: list[str]
    ) -> DataFrame:
        for nome_coluna in colunas:
            if nome_coluna in df.columns:
                df = df.withColumn(
                    nome_coluna,
                    trim(col(nome_coluna))
                )
        return df

    @staticmethod
    def converter_tipos(
        df: DataFrame,
        tipos: dict[str, str]
    ) -> DataFrame:
        for nome_coluna, tipo in tipos.items():
            if nome_coluna in df.columns:
                df = df.withColumn(
                    nome_coluna,
                    col(nome_coluna).cast(tipo)
                )
        return df

    @staticmethod
    def transformar_nan_em_null(
        df: DataFrame,
        colunas: list[str]
    ) -> DataFrame:
        for nome_coluna in colunas:
            if nome_coluna in df.columns:
                df = df.withColumn(
                    nome_coluna,
                    when(
                        isnan(col(nome_coluna)),
                        None
                    ).otherwise(col(nome_coluna))
                )
        return df

In [ ]:
def tratar_desempenho(df: DataFrame) -> DataFrame:
    colunas_double = [
        "taxa_alfabetizacao",
        "media_portugues",
        *[
            f"proporcao_aluno_nivel_{nivel}"
            for nivel in range(9)
        ]
    ]

    df = SilverProcessor.remover_duplicados(df)

    df = SilverProcessor.converter_tipos(
        df,
        {
            "ano": "int",
            "serie": "int",
            "rede": "int",
            **{
                coluna: "double"
                for coluna in colunas_double
            }
        }
    )

    df = SilverProcessor.transformar_nan_em_null(
        df,
        colunas_double
    )

    df = df.withColumn(
        "categoria_rede",
        when(col("rede") == 0, "Total")
        .when(col("rede") == 2, "Estadual")
        .when(col("rede") == 3, "Municipal")
        .when(col("rede") == 5, "Privada")
        .otherwise("Não informado")
    )

    return df

In [ ]:
silver_municipio = tratar_desempenho(
    bronze["municipio"]
)

silver_uf = tratar_desempenho(
    bronze["uf"]
)

In [ ]:
silver_municipio = (
    silver_municipio
    .withColumn(
        "id_municipio",
        col("id_municipio").cast("string")
    )
)

In [ ]:
COLUNAS_META = [
    f"meta_alfabetizacao_{ano}"
    for ano in range(2024, 2031)
]


def tratar_meta(df: DataFrame) -> DataFrame:
    tipos = {
        "ano": "int",
        "rede": "string",
        "taxa_alfabetizacao": "double",
        "percentual_participacao": "double",
        **{
            coluna: "double"
            for coluna in COLUNAS_META
        }
    }

    if "nivel_alfabetizacao" in df.columns:
        tipos["nivel_alfabetizacao"] = "double"

    df = SilverProcessor.remover_duplicados(df)
    df = SilverProcessor.converter_tipos(df, tipos)

    df = SilverProcessor.normalizar_textos(
        df,
        ["rede", "sigla_uf", "id_municipio"]
    )

    numericas = [
        "taxa_alfabetizacao",
        "percentual_participacao",
        *COLUNAS_META
    ]

    if "nivel_alfabetizacao" in df.columns:
        numericas.append("nivel_alfabetizacao")

    return SilverProcessor.transformar_nan_em_null(
        df,
        numericas
    )

In [ ]:
silver_meta_municipio = tratar_meta(
    bronze["meta_municipio"]
).withColumn(
    "id_municipio",
    col("id_municipio").cast("string")
)

silver_meta_uf = tratar_meta(
    bronze["meta_uf"]
)

silver_meta_brasil = tratar_meta(
    bronze["meta_brasil"]
)

In [ ]:
def tratar_alunos_agregados(
    df: DataFrame
) -> DataFrame:

    df = SilverProcessor.remover_duplicados(df)

    df = SilverProcessor.converter_tipos(
        df,
        {
            "ano": "int",
            "id_municipio": "string",
            "id_escola": "string",
            "serie": "string",
            "rede": "string",
            "total_registros": "long",
            "total_presentes": "long",
            "total_alfabetizados": "long",
            "total_cadernos_preenchidos": "long",
            "proficiencia_media": "double",
            "peso_medio": "double",
            "taxa_presenca": "double",
            "taxa_alfabetizacao_alunos": "double"
        }
    )

    df = SilverProcessor.normalizar_textos(
        df,
        [
            "id_municipio",
            "id_escola",
            "serie",
            "rede"
        ]
    )

    df = df.filter(
        col("ano").isNotNull()
        & col("id_municipio").isNotNull()
        & col("total_registros").isNotNull()
        & (col("total_registros") > 0)
    )

    df = df.withColumn(
        "taxa_preenchimento_caderno",
        when(
            col("total_registros") > 0,
            (
                col("total_cadernos_preenchidos")
                / col("total_registros")
            ) * 100
        )
    )

    return df

In [ ]:
silver_alunos = tratar_alunos_agregados(
    bronze["alunos_agregados"]
)

In [ ]:
silver = {
    "municipio": silver_municipio,
    "uf": silver_uf,
    "meta_municipio": silver_meta_municipio,
    "meta_uf": silver_meta_uf,
    "meta_brasil": silver_meta_brasil,
    "alunos_agregados": silver_alunos
}

In [ ]:
from pyspark.sql.functions import count

for nome, df in silver.items():
    total = df.count()
    duplicados = total - df.dropDuplicates().count()

    print("=" * 70)
    print(nome.upper())
    print(f"Registros: {total:,}")
    print(f"Colunas: {len(df.columns)}")
    print(f"Duplicados completos: {duplicados:,}")

MUNICIPIO
Registros: 23,995
Colunas: 20
Duplicados completos: 0
UF
Registros: 145
Colunas: 20
Duplicados completos: 0
META_MUNICIPIO
Registros: 10,704
Colunas: 17
Duplicados completos: 0
META_UF
Registros: 81
Colunas: 16
Duplicados completos: 0
META_BRASIL
Registros: 3
Colunas: 15
Duplicados completos: 0
ALUNOS_AGREGADOS
Registros: 79,273
Colunas: 18
Duplicados completos: 0


In [ ]:
silver_alunos.select(
    "taxa_presenca",
    "taxa_alfabetizacao_alunos",
    "taxa_preenchimento_caderno",
    "proficiencia_media"
).describe().show()

+-------+------------------+-------------------------+--------------------------+------------------+
|summary|     taxa_presenca|taxa_alfabetizacao_alunos|taxa_preenchimento_caderno|proficiencia_media|
+-------+------------------+-------------------------+--------------------------+------------------+
|  count|             79273|                    79273|                     79273|             79273|
|   mean| 87.47459695180825|        51.68691284971745|         87.44531092882042|               NaN|
| stddev|12.310878036348544|        22.94093535364086|        12.313995657016807|               NaN|
|    min|               0.0|                      0.0|                       0.0|       601.0632443|
|    max|             100.0|                    100.0|                     100.0|               NaN|
+-------+------------------+-------------------------+--------------------------+------------------+



In [ ]:
silver_alunos.filter(
    (col("taxa_presenca") < 0)
    | (col("taxa_presenca") > 100)
    | (col("taxa_alfabetizacao_alunos") < 0)
    | (col("taxa_alfabetizacao_alunos") > 100)
).count()

0

In [ ]:
for nome, df in silver.items():
    caminho = f"{SILVER_PATH}/{nome}"

    writer = (
        df.write
        .mode("overwrite")
    )

    if "ano" in df.columns:
        writer = writer.partitionBy("ano")

    writer.parquet(caminho)

    print(f"Silver gravada: {nome}")

Silver gravada: municipio
Silver gravada: uf
Silver gravada: meta_municipio
Silver gravada: meta_uf
Silver gravada: meta_brasil
Silver gravada: alunos_agregados
